# W8A16 GPTQ - 성능 최대 보존 버전

## 전략
- **W4A16 → W8A16**: 8비트 양자화로 성능 손실 최소화
- **캘리브레이션 극대화**: 샘플 수 & 시퀀스 길이 최대
- **group_size=64**: 더 세밀한 양자화 (128 → 64)

## 예상 크기
| 항목 | W4A16 (기존) | W8A16 (본 버전) |
|------|-------------|----------------|
| 모델 | ~1.4 GB | ~2.5 GB |
| zip | ~0.88 GB | ~2.2 GB |
| 성능 보존 | 보통 | 최고 |

---

# 1. Import

In [6]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU 모드로 실행됩니다")
print("\n✅ Import 완료!")

PyTorch: 2.9.1
CUDA: False
⚠️ CPU 모드로 실행됩니다

✅ Import 완료!


# 2. 설정 (W8A16 + 파라미터 최대화)

In [7]:
# ============================================================================
# 모델 설정
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = "./model"

# ============================================================================
# 데이터셋 설정
# ============================================================================
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# ⭐ 캘리브레이션 설정 (최대화)
# ============================================================================
NUM_CALIBRATION_SAMPLES = 1024   # 256 → 1024 (4배)
MAX_SEQUENCE_LENGTH = 2048       # 512 → 2048 (4배)

# ============================================================================
# ⭐ 양자화 설정 (W8A16)
# ============================================================================
SCHEME = "W8A16"                 # 8비트 양자화 (성능 최대 보존)
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

# ============================================================================
# ⭐ GPTQ 최적화 파라미터 (최대)
# ============================================================================
BLOCK_SIZE = 128                 # Marlin 커널 호환
GROUP_SIZE = 64                  # 128 → 64 (더 세밀한 양자화)
DAMPENING_FRAC = 0.001           # Hessian 안정화
ACTORDER = "weight"              # 가중치 기반 정렬 (정확도↑)

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("W8A16 GPTQ - 성능 최대 보존 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SCHEME: {SCHEME}")
print("---")
print("⭐ 캘리브레이션 (최대화):")
print(f"  SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"  MAX_LEN: {MAX_SEQUENCE_LENGTH}")
print("⭐ GPTQ 파라미터:")
print(f"  BLOCK_SIZE: {BLOCK_SIZE}")
print(f"  GROUP_SIZE: {GROUP_SIZE}")
print(f"  DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"  ACTORDER: {ACTORDER}")
print("=" * 60)

W8A16 GPTQ - 성능 최대 보존 설정
MODEL_ID: ./open/base_model
SCHEME: W8A16
---
⭐ 캘리브레이션 (최대화):
  SAMPLES: 1024
  MAX_LEN: 2048
⭐ GPTQ 파라미터:
  BLOCK_SIZE: 128
  GROUP_SIZE: 64
  DAMPENING_FRAC: 0.001
  ACTORDER: weight


# 3. 모델 로드

In [8]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 모델/토크나이저 로드 완료


# 4. 데이터셋 로드

In [9]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print(f"[INFO] 시퀀스 최대 길이: {MAX_SEQUENCE_LENGTH}")
print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 1024
[INFO] 시퀀스 최대 길이: 2048
[INFO] 데이터 전처리 완료


# 5. GPTQ 양자화 (W8A16)

In [10]:
print("[INFO] GPTQ W8A16 양자화 시작")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - block_size: {BLOCK_SIZE}")
print(f"  - group_size: {GROUP_SIZE}")
print(f"  - actorder: {ACTORDER}")
print(f"  - dampening_frac: {DAMPENING_FRAC}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드\n")
else:
    print("\n⏳ CPU 모드: 오래 걸릴 수 있음\n")

recipe = [
    GPTQModifier(
        config_groups={
            "group_0": {
                "targets": TARGETS,
                "weights": {
                    "num_bits": 8,
                    "type": "int",
                    "symmetric": True,
                    "strategy": "group",
                    "group_size": GROUP_SIZE,
                },
            }
        },
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ W8A16 양자화 완료!")

[INFO] GPTQ W8A16 양자화 시작
  - scheme: W8A16
  - samples: 1024
  - max_len: 2048
  - block_size: 128
  - group_size: 64
  - actorder: weight
  - dampening_frac: 0.001

⏳ CPU 모드: 오래 걸릴 수 있음



Tokenizing:   0%|          | 0/1024 [00:00<?, ? examples/s]

2026-02-15T22:30:28.333861+0900 | reset | INFO - Compression lifecycle reset
2026-02-15T22:30:28.334980+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-15T22:30:28.354651+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-15T22:30:28.354918+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-15T22:30:28.360190+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0215 22:30:28.382000 3054 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [05:17<00:00,  3.22it/s]

2026-02-15T22:35:47.386692+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 1024 samples


...

(30/31): Calibrating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [05:22<00:00,  3.18it/s]

2026-02-16T02:36:06.303695+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 1024 samples


2026-02-16T02:36:06.605608+0900 | compress | METRIC - time 0.30s
2026-02-16T02:36:06.605976+0900 | compress | METRIC - error 21.39
2026-02-16T02:36:06.609015+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-16T02:36:06.609382+0900 | compress | METRIC - Compressed module size: 17.104896 MB
2026-02-16T02:36:06.610733+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 1024 samples
2026-02-16T02:36:06.792038+0900 | compress | METRIC - time 0.18s
2026-02-16T02:36:06.792496+0900 | compress | METRIC - error 5.74
2026-02-16T02:36:06.793244+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-16T02:36:06.793471+0900 | compress | METRIC - Compressed module size: 4.276224 MB
2026-02-16T02:36:06.794100+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 1024 samples
2026-02-16T02:36:06.975834+0900 | compress | METRIC - time 0.18s
2026-02-16T02:36:

(31/31): Propagating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [00:00<00:00, 1061.62it/s]


2026-02-16T02:38:50.201550+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-16T02:38:50.224647+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] GPTQ W8A16 양자화 완료!


# 6. 모델 저장

In [11]:
print("[INFO] 모델 저장 중...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  W8A16 모델:    {quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-16T02:38:50.378478+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:04, 43.72it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1975.9 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  W8A16 모델:    1.99 GB
  압축률:        77.6%


# 7. 제출 파일 생성

In [12]:
zip_name = "submit_w8a16"
print(f"[INFO] {zip_name}.zip 생성 중...")

if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

print("\n" + "=" * 60)
print("제출 파일 구조")
print("=" * 60)
print(f"{zip_name}.zip")
print(f"└── model/")
for f in sorted(os.listdir(OUT_DIR))[:5]:
    print(f"    ├── {f}")
print("    └── ...")
print("=" * 60)

[INFO] submit_w8a16.zip 생성 중...
[INFO] 생성 완료: submit_w8a16.zip (1.44 GB)
✅ 용량 제한 충족 (≤ 10GB)

제출 파일 구조
submit_w8a16.zip
└── model/
    ├── chat_template.jinja
    ├── config.json
    ├── generation_config.json
    ├── merges.txt
    ├── model.safetensors
    └── ...


---

# W8A16 vs W4A16 비교

| 항목 | W4A16 (06번) | W8A16 (본 버전) |
|------|-------------|----------------|
| 비트 | 4-bit | 8-bit |
| 캘리브레이션 샘플 | 256 | 1024 |
| 시퀀스 길이 | 512 | 2048 |
| group_size | 128 | 64 |
| 모델 크기 | ~1.4 GB | ~2.5 GB |
| PerfNorm | ~0.95 | ~0.99 |
| SpeedNorm | 높음 (Marlin) | 중간 |

---